In [4]:
import pandas as pd
import numpy as np
import pickle

file = 'DATAFILES/data_economics.xlsx'
xl = pd.ExcelFile(file)
print(xl.sheet_names)
d = {} # your dict.
for sheet in xl.sheet_names:
    d[f'{sheet}']= pd.read_excel(xl,sheet_name=sheet)
    pickle.dump(d[f'{sheet}'], open('HOME_PICKLE_FILES/' + sheet + '.pkl','wb'))

### Uploading pkl files

In [5]:
idatalist = pd.read_pickle("HOME_PICKLE_FILES/idatalist.pkl")
inames = pd.read_pickle("HOME_PICKLE_FILES/ieconames.pkl")
datalist = pd.read_pickle("HOME_PICKLE_FILES/datalist.pkl")
names = pd.read_pickle("HOME_PICKLE_FILES/econames.pkl")
institutions = pd.read_pickle("HOME_PICKLE_FILES/institutions.pkl")
journals = pd.read_pickle("HOME_PICKLE_FILES/journals.pkl")
inewforscores = institutions.copy()
newforscores = journals.copy()

### The reputed citation algorithm

In [6]:
def updatescores(datalist, idatalist, journals, institutions, newforscores,inewforscores,names,inames):
    acumulador = [] #to append the acronym (name) of the journal and the score from the suma function
    iacumulador = [] #to append the acronym (name) of the journal and the score from the suma function
    newforscores['journal_score'] =  newforscores['journal_score'].fillna(.1)
    renewforscores = newforscores[['eco_incites','journal_score']].copy()
    for k,name in zip(range(len(datalist)),names):
        data = datalist[k]
        #data = df[['eco_incites','eco_institution','ninst']]
        scores = data.merge(inewforscores, on='eco_institution',how='left')
        scores['iscore'] = scores['iscore'].fillna(.1)
        scores = scores.merge(newforscores, on='eco_incites',how='left')
        scores['journal_score'] = scores['journal_score'].fillna(.1)
        suma = (scores['journal_score']*scores['iscore']/scores['ninst']).sum()
        acumulador.append([name,suma])
    for k,name in zip(range(len(idatalist)),inames):
        data = idatalist[k]
        #data = df[['eco_incites','eco_institution','ninst']]
        scores = data.merge(inewforscores, on='eco_institution',how='left')
        scores = scores.merge(renewforscores, on='eco_incites',how='left')
        scores['instscores'] = np.where(scores['acr'].isin(inames),scores['iscore'],.1)
        suma = (scores['journal_score']*scores['instscores']/scores['ninst']).sum()
        iacumulador.append([name,suma])
    acumdata =  pd.DataFrame(acumulador, columns=['acr','suma'])#stores the institutional scores at iteration l
    usejournals = journals.merge(acumdata,on='acr',how='left')
    acumdata =  pd.DataFrame(iacumulador, columns=['acr','suma'])#stores the institutional scores at iteration l
    useinstitutions = institutions.merge(acumdata,on='acr',how='left')
    usejournals['suma'] = usejournals['suma']/usejournals['pub']#the suma return is diveded by the number of articles
    useinstitutions['suma'] = useinstitutions['suma']/useinstitutions['pub']#the suma return is diveded by the number of articles
    useinstitutions['suma'] = 10*np.sqrt(useinstitutions['suma']/np.linalg.norm(useinstitutions['suma']))
    usejournals['suma'] = 10*np.sqrt(usejournals['suma']/np.linalg.norm(usejournals['suma']))
    journal_sum = usejournals['suma'].sum()
    inst_sum = useinstitutions['suma'].sum()
    newforscores = newforscores.merge(usejournals[['acr','suma']],on='acr',how='left')#add the column l to newforscores      
    newforscores['journal_score'] = newforscores['suma']#update the journal_scores with the current values of the iteration
    newforscores = newforscores[['eco_incites', 'acr', 'pub','journal_score']]
    inewforscores = inewforscores.merge(useinstitutions[['acr','suma']],on='acr',how='left')#add the column l to newforscores      
    inewforscores['iscore'] = inewforscores['suma']#update the journal_scores with the current values of the iteration
    inewforscores = inewforscores[['eco_institution', 'acr', 'pub','iscore']]
    totalsum = journal_sum + inst_sum
    return newforscores,  inewforscores, totalsum

In [ ]:
oldtotalsum = 1; totalsum = 5
while np.abs(oldtotalsum - totalsum)>1:
    oldtotalsum = totalsum
    newforscores,  inewforscores, totalsum = updatescores(datalist, idatalist, journals, institutions, newforscores,inewforscores,names,inames)
    print('totalsum =', totalsum)

totalsum = 1630.3541002439363
totalsum = 1406.896519291267
totalsum = 1332.643901166632
totalsum = 1304.3132096480153


In [8]:
renewforscores = newforscores.copy()
irenewforscores = inewforscores.copy()
filename = 'RESULTS/fractional_inst_journal_scores.xlsx'
renewforscores['journal_score'] = np.maximum(np.round(10*np.sqrt(renewforscores['journal_score']/renewforscores['journal_score'].max()),1),0.1)
renewforscores = renewforscores.sort_values(by=['journal_score','eco_incites'],ascending=[False, True])
irenewforscores['iscore'] = np.maximum(np.round(10*irenewforscores['iscore']/irenewforscores['iscore'].max(),1),.5)
irenewforscores = irenewforscores.sort_values(by=['iscore','eco_institution'],ascending=[False, True])
with pd.ExcelWriter(filename) as writer:
    renewforscores.to_excel(writer,sheet_name='journals')
    irenewforscores.to_excel(writer,sheet_name='institutions')

In [ ]:
print('The end')